In [1]:
import lamindb as ln
import os

# --- 1. CONFIGURATION DES CHEMINS RELATIFS ---
# On remonte de jupyter/notebooks/ vers la racine vitessce/ pour trouver data/
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "data"))
storage_path = os.path.join(BASE_DIR, "lamin_storage")

print(f"🏠 Racine des données détectée : {BASE_DIR}")
print(f"📦 Chemin du stockage LaminDB : {storage_path}")

# --- 2. PRÉPARATION DU DOSSIER ---
if not os.path.exists(storage_path):
    os.makedirs(storage_path)
    print("📁 Dossier lamin_storage créé.")

# --- 3. INITIALISATION OU CHARGEMENT ---
try:
    # On tente d'initialiser l'instance
    # db="sqlite" est souvent explicite, mais optionnel
    ln.setup.init(storage=storage_path, name="7Hills_Project")
    print("✅ LaminDB a été initialisé avec succès !")
except Exception as e:
    # Si l'instance existe déjà, on la charge
    if "already exists" in str(e) or "InstanceExists" in str(e):
        print("ℹ️ L'instance existe déjà. Chargement en cours...")
        ln.setup.load("7Hills_Project")
        print("✅ LaminDB est prêt et chargé !")
    else:
        print(f"❌ Erreur inattendue lors de l'initialisation : {e}")

try:
    ln.track("H0Qr7glGraLp") # Utilise l'UID fixe suggéré par Lamin
    print("🚀 Script tracé avec UID fixe.")
except Exception:
    print("⚠️ Tracking impossible.")

🏠 Racine des données détectée : /home/jovyan/work/data
📦 Chemin du stockage LaminDB : /home/jovyan/work/data/lamin_storage
📁 Dossier lamin_storage créé.
! using anonymous user (to identify, call: lamin login)
→ initialized lamindb: anonymous/7Hills_Project
✅ LaminDB a été initialisé avec succès !
→ created Transform('H0Qr7glGraLp0000', key='011-Init_LaminDB.ipynb'), started new Run('QW3BT3aRGWOXbyuY') at 2026-02-12 16:55:16 UTC
→ notebook imports: lamindb==2.1.2 pandas==2.1.1
🚀 Script tracé avec UID fixe.


In [2]:
import pandas as pd
import lamindb as ln
import os

# --- 0. TRAÇABILITÉ (Fixé avec l'UID suggéré par LaminDB) ---
# Cela évite de créer des doublons si tu renommes le fichier
ln.track("H0Qr7glGraLp")

# --- 1. CHEMINS D'ACCÈS RELATIFS ---
# On remonte de jupyter/notebooks/ vers la racine vitessce/ pour trouver data/
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "data"))

CSV_PATH = os.path.join(BASE_DIR, "liste_patients_DATA.csv")
H5AD_DIR = os.path.join(BASE_DIR, "script01_adatas")
ZARR_DIR = os.path.join(BASE_DIR, "script01_zarrs")

print(f"🏠 Racine des données : {BASE_DIR}")

# --- 2. CHARGEMENT DES MÉTADONNÉES ---
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f"📊 CSV chargé : {len(df)} lignes trouvées.")
else:
    print(f"❌ Erreur : CSV introuvable à {CSV_PATH}")
    # Optionnel : stopper ici si le CSV manque
    raise FileNotFoundError(CSV_PATH)

# --- 3. FONCTION POUR INDEXER UN FICHIER ---
def register_artifact(file_path, description, labels_dict):
    # Vérifier si l'artefact existe déjà
    artifact = ln.Artifact.filter(description=description).first()
    
    if artifact is None:
        # Création de l'artefact
        artifact = ln.Artifact(file_path, description=description)
        artifact.save()
    
    # Ajout des ULabels (Patient, Statut, Région)
    for feature_name, label_value in labels_dict.items():
        if pd.isna(label_value): continue
        
        lbl = ln.ULabel.filter(name=str(label_value)).first()
        if lbl is None:
            lbl = ln.ULabel(name=str(label_value), description=f"Metadata: {feature_name}")
            lbl.save()
        
        # Note : On utilise .add() car ton manager n'a pas .ulabels
        artifact.labels.add(lbl)
    
    artifact.save()
    return artifact

# --- 4. BOUCLE D'INDEXATION ---
for _, row in df.iterrows():
    h5ad_name = row['File_Name']
    patient_id = row['Patient_ID']
    status = row['Statut']
    
    # Détection de la région basée sur le nom du fichier
    region = "Core" if "Core" in h5ad_name else "Edge" if "Edge" in h5ad_name else "Other"
    
    meta_labels = {
        "Patient_ID": patient_id,
        "Tumor_Status": status,
        "Region": region
    }

    # A. Indexation du H5AD
    h5ad_path = os.path.join(H5AD_DIR, h5ad_name)
    if os.path.exists(h5ad_path):
        register_artifact(h5ad_path, f"Source: {h5ad_name}", meta_labels)
        print(f"✅ H5AD : {h5ad_name}")
    else:
        print(f"⚠️ H5AD manquant : {h5ad_name}")

    # B. Indexation du ZARR
    zarr_name = h5ad_name.replace(".h5ad", ".zarr")
    zarr_path = os.path.join(ZARR_DIR, zarr_name)
    if os.path.exists(zarr_path):
        register_artifact(zarr_path, f"Viz: {zarr_name}", meta_labels)
        print(f"✅ ZARR : {zarr_name}")
    else:
        print(f"⚠️ ZARR manquant : {zarr_name}")

print("\n🚀 Indexation terminée et sécurisée dans LaminDB.")

→ loaded Transform('H0Qr7glGraLp0000', key='011-Init_LaminDB.ipynb'), re-started Run('QW3BT3aRGWOXbyuY') at 2026-02-12 16:55:19 UTC
→ notebook imports: lamindb==2.1.2 pandas==2.1.1
🏠 Racine des données : /home/jovyan/work/data
📊 CSV chargé : 40 lignes trouvées.
! calling anonymously, will miss private instances
✅ H5AD : adata_AA1_Core.h5ad
✅ ZARR : adata_AA1_Core.zarr
✅ H5AD : adata_AA1_Edge.h5ad
✅ ZARR : adata_AA1_Edge.zarr
✅ H5AD : adata_AA2_Edge.h5ad
✅ ZARR : adata_AA2_Edge.zarr
✅ H5AD : adata_AA2__Core.h5ad
✅ ZARR : adata_AA2__Core.zarr
✅ H5AD : adata_AA3_Core.h5ad
✅ ZARR : adata_AA3_Core.zarr
✅ H5AD : adata_AA3_Edge.h5ad
✅ ZARR : adata_AA3_Edge.zarr
✅ H5AD : adata_Control_1.h5ad
✅ ZARR : adata_Control_1.zarr
✅ H5AD : adata_Control_2.h5ad
✅ ZARR : adata_Control_2.zarr
✅ H5AD : adata_GBM1-Core.h5ad
✅ ZARR : adata_GBM1-Core.zarr
✅ H5AD : adata_GBM1-Edge.h5ad
✅ ZARR : adata_GBM1-Edge.zarr
✅ H5AD : adata_GBM10-Core.h5ad
✅ ZARR : adata_GBM10-Core.zarr
✅ H5AD : adata_GBM10-Edge.h5ad
✅ ZA